<a href="https://colab.research.google.com/github/carlosvasquez3/modelo-rendimiento-agricola/blob/main/B_preparacion_datos_EVA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preparación de datos EVA

En este notebook se realiza la preparación de la base de datos de las Evaluaciones Agropecuarias Municipales EVA para construir el modelo de predicción del rendimiento agrícola.

Se parte de la base original y se realizan las validaciones, selección de variables, limpieza y transformaciones necesarias para dejar los datos listos para el modelamiento.

## Carga de datos

In [1]:
# Importe de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Para trabajar con archivos (ZIP) desde GitHub
import requests
import zipfile
from io import BytesIO

# Para evitar mensajes innecesarios durante el análisis
import warnings
warnings.filterwarnings("ignore")

In [2]:
# URL del archivo ZIP almacenado en GitHub
url = "https://raw.githubusercontent.com/carlosvasquez3/modelo-rendimiento-agricola/main/A_base_datos_original_EVA.zip"

# Descargar el archivo ZIP
respuesta = requests.get(url)

# Abrir el ZIP y tomar el único archivo que contiene
with zipfile.ZipFile(BytesIO(respuesta.content)) as zip_file:
    archivo = zip_file.namelist()[0]

    # Leer el archivo CSV y guardarlo en un DataFrame
    df = pd.read_csv(zip_file.open(archivo))

# Visualizar los primeros registros
df.head()

,Código Dane departamento,Departamento,Código Dane municipio,Municipio,Grupo cultivo,Subgrupo,Cultivo,Desagregación cultivo,Año,Periodo,Área sembrada,Área cosechada,Producción,Rendimiento,Ciclo del cultivo,Estado físico del cultivo,Código del cultivo,Nombre científico del cultivo
0,5,Antioquia,5001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2021,2021A,"8,00","8,00","128,00","16,00",Transitorio,En fresco,1050600,Apium graveolens
1,5,Antioquia,5001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2021,2021B,"8,00","8,00","152,00","19,00",Transitorio,En fresco,1050600,Apium graveolens
2,5,Antioquia,5001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2022,2022A,"8,00","8,00","127,44","15,93",Transitorio,En fresco,1050600,Apium graveolens
3,5,Antioquia,5001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2022,2022B,"8,00","8,00","152,00","19,00",Transitorio,En fresco,1050600,Apium graveolens
4,5,Antioquia,5001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2023,2023A,"8,00","7,00","133,00","19,00",Transitorio,En fresco,1050600,Apium graveolens


## Revisión incial de la base de datos

In [3]:
# Dimensiones de la base
print("Filas y columnas", df.shape)

Filas y columnas (166732, 18)


In [4]:
# Tipos de datos
print("\nTipos de datos")
print(df.dtypes)


Tipos de datos
Código Dane departamento          int64
Departamento                     object
Código Dane municipio             int64
Municipio                        object
Grupo cultivo                    object
Subgrupo                         object
Cultivo                          object
Desagregación cultivo            object
Año                               int64
Periodo                          object
Área sembrada                    object
Área cosechada                   object
Producción                       object
Rendimiento                      object
Ciclo del cultivo                object
Estado físico del cultivo        object
Código del cultivo                int64
Nombre científico del cultivo    object
dtype: object


In [5]:
# Valores nulos
print("\nValores nulos")
print(df.isnull().sum())


Valores nulos
Código Dane departamento         0
Departamento                     0
Código Dane municipio            0
Municipio                        0
Grupo cultivo                    0
Subgrupo                         0
Cultivo                          0
Desagregación cultivo            0
Año                              0
Periodo                          0
Área sembrada                    0
Área cosechada                   0
Producción                       0
Rendimiento                      0
Ciclo del cultivo                0
Estado físico del cultivo        0
Código del cultivo               0
Nombre científico del cultivo    0
dtype: int64


La base contiene 166.732 registros y 18 variables. No se encontraron valores nulos en ninguna de las columnas.

Durante la revisión de los tipos de datos se identificó que las variables Área sembrada, Área cosechada, Producción y Rendimiento fueron cargadas como texto, por lo que será necesario convertirlas a formato numérico durante la preparación de los datos.

In [6]:
# Variables que deben ser numéricas
columnas_numericas = [
    "Área sembrada",
    "Área cosechada",
    "Producción",
    "Rendimiento"
]

# Convertir únicamente las variables que todavía están como texto
for columna in columnas_numericas:
    if df[columna].dtype == "object":
        df[columna] = (
            df[columna]
            .str.replace(".", "", regex=False)
            .str.replace(",", ".", regex=False)
            .astype(float)
        )

# Revisar los tipos de datos
print(df[columnas_numericas].dtypes)

Área sembrada     float64
Área cosechada    float64
Producción        float64
Rendimiento       float64
dtype: object


Durante la revisión inicial se identificó que Área sembrada, Área cosechada, Producción y Rendimiento estaban almacenadas como variables de tipo object, aunque por su naturaleza corresponden a variables numéricas.

Por esta razón se realizó la conversión de estas columnas a float64, conservando sus valores decimales. Después del ajuste, la base cuenta con 8 variables almacenadas como numéricas y 10 como categóricas.

También se verificó que la base contiene 166.732 registros y que no presenta valores nulos.

## Selección de variables para el modelo

In [8]:
# Selección de variables para el modelo
df_modelo = df[
    ["Departamento", "Grupo cultivo", "Periodo", "Rendimiento"]
].copy()

df_modelo.head()

,Departamento,Grupo cultivo,Periodo,Rendimiento
0,Antioquia,Hortalizas,2021A,16.00
1,Antioquia,Hortalizas,2021B,19.00
2,Antioquia,Hortalizas,2022A,15.93
3,Antioquia,Hortalizas,2022B,19.00
4,Antioquia,Hortalizas,2023A,19.00


Después de revisar la estructura de la base se seleccionaron únicamente las variables que serán utilizadas en el modelo. Se conservaron Departamento, Grupo cultivo, Periodo y Rendimiento.

Las variables Producción y Área cosechada fueron excluidas porque participan directamente en el cálculo del Rendimiento y podrían generar fuga de información. Área sembrada se eliminó con el fin de simplificar el modelo.

También se eliminaron variables como Municipio, Cultivo, Subgrupo y Desagregación debido a la gran cantidad de categorías que podrían generar. Los códigos fueron descartados porque funcionan principalmente como identificadores.

De esta manera, el conjunto de datos queda reducido a tres variables de entrada categóricas y una variable numérica de salida.

In [10]:
# Revisamos tipo
print(df_modelo.dtypes)

Departamento      object
Grupo cultivo     object
Periodo           object
Rendimiento      float64
dtype: object


## ydata-profiling

In [11]:
!pip install ydata-profiling -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.5/682.5 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.3 MB/s eta 0:00:00


In [12]:
# Importar la librería
from ydata_profiling import ProfileReport

# Generar el reporte de perfilamiento
reporte = ProfileReport(
    df_modelo,
    title="Perfilamiento de datos EVA"
)

# Guardar el reporte en formato HTML
reporte.to_file("C_reporte_ydata_EVA.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 4/4 [00:00<00:00,  6.09it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

## Revisión de duplicados

Se revisan los registros duplicados sobre la base original para determinar si las repeticiones identificadas durante el perfilamiento corresponden realmente a filas completamente iguales.

In [13]:
# Contar registros completamente duplicados en la base original
duplicados_original = df.duplicated().sum()

# Número de registros duplicados
print("Duplicados exactos en la base original", duplicados_original)

Duplicados exactos en la base original 0


### Revisión de registros duplicados

Durante el perfilamiento se identificaron combinaciones repetidas dentro de las cuatro variables seleccionadas. Para comprobar si correspondían a duplicados reales, se realizó una validación sobre las 18 variables de la base original.

La revisión mostró que no existen registros completamente duplicados. Por esta razón, no se realizó ninguna eliminación por duplicidad y se conservaron todos los registros disponibles.

## Codificación de variables

Las variables Departamento, Grupo cultivo y Periodo son categóricas, por lo que será necesario transformarlas a una representación numérica antes de entrenar los modelos.

Para este proceso se utilizará One Hot Encoding, ya que las categorías no presentan un orden natural entre sí.

In [14]:
# Importar herramientas para la codificación
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Variables de entrada
X = df_modelo[
    ["Departamento", "Grupo cultivo", "Periodo"]
]

# Variable objetivo
y = df_modelo["Rendimiento"]

# Variables categóricas que serán codificadas
columnas_categoricas = [
    "Departamento",
    "Grupo cultivo",
    "Periodo"
]

# Configurar la codificación One Hot Encoding
preprocesador = ColumnTransformer(
    transformers=[
        (
            "categoricas",
            OneHotEncoder(handle_unknown="ignore"),
            columnas_categoricas
        )
    ]
)

# Revisar las dimensiones
print("Variables de entrada", X.shape)
print("Variable objetivo", y.shape)

Variables de entrada (166732, 3)
Variable objetivo (166732,)


### Análisis

Se definieron Departamento, Grupo cultivo y Periodo como variables de entrada y Rendimiento como variable objetivo.

Para las variables categóricas se configuró One Hot Encoding, permitiendo que cada categoría sea representada mediante variables binarias sin establecer un orden numérico entre ellas.

La codificación se aplicará posteriormente sobre los datos de entrenamiento para evitar que información del conjunto de prueba intervenga durante la preparación del modelo.